# Model Deployment

Tugas kelima menutup rangkaian klasifikasi dengan menerbitkan model menjadi aplikasi yang bisa dipakai orang lain: pengguna menempelkan link berita atau menempelkan teksnya, lalu aplikasi menjawab apakah berita itu sport atau finance. Model yang dipakai hanya lengan skip-gram yang menang di tugas Modeling, dikemas bersama pipeline pembersihan yang sama persis dengan data latih supaya prediksi di aplikasi tidak melenceng dari hasil validasi. Data dan model semua berasal dari penyimpanan tahap sebelumnya; tidak ada crawling baru di sini.

## Persiapan Model

Empat artefak hasil tugas Modeling disalin ke folder distribusi `deploy/models/`: model skip-gram, estimator naive bayes, urutan kelas, dan daftar nama yang dilindungi saat normalisasi slang. Kelengkapan artefak dicek langsung dari folder yang akan terkirim ke platform deployment.

In [1]:
from pathlib import Path

DEPLOY = Path('..') / '..' / 'deploy'
files = sorted((DEPLOY / 'models').iterdir())
print('\n'.join(f'{f.name} ({f.stat().st_size:,} bytes)' for f in files))

gnb.joblib (2,415 bytes)
labels.json (20 bytes)
protected.json (12,981 bytes)
word2vec.model (1,656,252 bytes)


## Pipeline Pembersihan

Aplikasi memutar transformasi teks yang identik dengan data latih: normalisasi ascii (aksen, tanda baca, simbol non-latin), artefak format, aturan angka, pemecahan kalimat, lalu per-token lowercase dan normalisasi slang dengan proteksi nama diri. Token yang tidak ada di kosakata skip-gram dilewati, vektor sisanya dirata-rata, dan hasilnya masuk ke estimator naive bayes. Dua contoh berikut memperlihatkan jawaban aplikasi untuk satu kalimat tiap kelas.

In [2]:
import sys

sys.path.insert(0, str(DEPLOY))

from serve import classify, describe

cases = [
    'Imbal hasil obligasi negara menguat di tengah rilis data neraca dagang',
    'Gol tunggal di babak kedua membawa tuan rumah menang atas sang juara bertahan',
]
for text in cases:
    label, scores = classify(text)
    stats = describe(text)
    print(
        f"{label:8s} sport {scores['sport']:.3f} finance {scores['finance']:.3f} "
        f"| {stats['tokens']} token, {stats['words in vocabulary']} dikenal"
    )

finance  sport 0.000 finance 1.000 | 11 token, 9 dikenal
sport    sport 1.000 finance 0.000 | 13 token, 10 dikenal


## Parity Check

Selain contoh manual, pipeline servis diuji ulang pada data uji yang sama dengan tugas Modeling (splitserasi 160/40, seed 42, stratify label). Akurasi yang dihasilkan lewat fungsi `classify` di aplikasi harus sama dengan hasil validasi Modeling.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

DATA = Path('..') / '..' / 'data' / 'Web-Mining'
raw = pd.read_csv(DATA / 'crawling_detik.csv')
_, test_idx = train_test_split(
    list(range(len(raw))),
    test_size=0.2,
    random_state=42,
    stratify=raw['label'],
)
hits = 0
for i in test_idx:
    label, _ = classify(raw['isi_berita'].iloc[i])
    hits += int(label == raw['label'].iloc[i])
print(f'akurasi uji lewat pipeline aplikasi: {hits}/{len(test_idx)} = {hits / len(test_idx):.3f}')

akurasi uji lewat pipeline aplikasi: 39/40 = 0.975


## Publish Space

Aplikasi dipasang di Hugging Face Space atas nama Rhindottire memakai Gradio pada perangkat ZeroGPU (Python 3.12) — rute gratis setelah Hugging Face menarik hosting Gradio biasa ke belakang biaya. ZeroGPU mensyaratkan minimal satu fungsi berdekorator `@spaces.GPU`; karena model ini murni CPU, dekoratornya ditaruh di fungsi klasifikasi. Status Space dicek lewat API publiknya.

In [4]:
import json
import urllib.request

with urllib.request.urlopen(
    'https://huggingface.co/api/spaces/Rhindottire/News-Classifier',
    timeout=30,
) as response:
    info = json.load(response)
print('stage:', info['runtime']['stage'])
print('url:', 'https://' + info['subdomain'] + '.hf.space')

stage: RUNNING
url: https://rhindottire-news-classifier.hf.space


## Cara Pakai

Buka URL Space di atas, lalu pilih salah satu masukan: tempel link artikel (isi halaman diambil dengan trafilatura tanpa bergantung struktur HTML, jeda satu detik antar permintaan, user agent jujur, dan berhenti bila situs menjawab 403/429) atau tempel teks berita langsung. Tekan Klasifikasi; jawaban menampilkan kategori sport/finance, skor kedua kelas, dan ringkasan jumlah kalimat, token, serta cakupan kosakata.

## Conclusion

Aplikasi klasifikasi berita berjalan di alamat publik Hugging Face Space dan menjawab kelas sport/finance dari URL maupun teks tempel. Distribusi ini tidak membangun ulang model: ia memutar ulang transformasi yang sama persis dengan data latih, dan parity pada data uji menunjukkan akurasi yang sama dengan tugas Modeling, sehingga prediksi di web setara dengan hasil validasi.